# [실습 1] 학습 안정성 디버깅과 체크포인트 복구

## 실습 개요

> "loss가 튀는 긴 학습을 어떻게 관측하고 복구 가능한 흐름으로 만들 수 있을까?"

이번 실습에서는 작은 PyTorch 분류 모델을 사용해 loss, gradient norm, learning rate를 함께 관찰합니다. 강의에서 다룬 gradient clipping, warmup 이후 decay scheduler, checkpoint 저장과 재개 흐름을 하나의 학습 루프로 연결합니다.

실제 대형 Transformer 학습은 훨씬 오래 걸리지만, 안정성을 확인하는 절차는 작은 모델에서도 동일합니다. 문제가 생겼을 때 감으로 설정을 바꾸지 않고 관측 가능한 값으로 원인을 좁히는 연습에 초점을 둡니다.

## 실습 목표

1. 학습 설정을 JSON 파일에서 읽어 실험 조건을 고정한다.
2. loss와 gradient norm을 기록해 학습 이상 징후를 확인한다.
3. gradient clipping과 learning rate scheduler를 학습 루프에 연결한다.
4. checkpoint를 저장하고 다시 불러와 학습을 이어갈 수 있는 구조를 만든다.
5. [TODO] 코드를 완성해 안정적인 학습 step을 구현한다.

## 데이터 출처

- 데이터: 실습용 2차원 분류 데이터 직접 생성
- 설정 파일: `config/training_config.json`

## 목차

1. 설정과 라이브러리 준비
2. 실습용 데이터 생성
3. 모델과 관측 함수 구성
4. 기본 학습 루프 실행
5. [TODO] 안정화 학습 step 구현
6. checkpoint 저장과 복구

---
## 1. 설정과 라이브러리 준비

### 실험 설정을 파일로 분리하기

긴 학습에서 learning rate, warmup step, clipping 기준, seed는 단순한 옵션이 아니라 실험의 일부입니다. 코드 안에 흩어져 있으면 나중에 loss가 튀었을 때 어떤 조건에서 문제가 생겼는지 되짚기 어렵습니다.

설정을 JSON으로 분리해 두면 같은 코드를 유지한 채 실험 조건만 바꿔 비교할 수 있습니다. 실제 Transformer 학습에서도 checkpoint와 함께 설정 파일을 보관해야, 며칠 뒤 이어서 학습하거나 다른 환경에서 재현할 때 기준이 흔들리지 않습니다.

In [29]:
import json
import math
import random
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F

config_path = Path('config/training_config.json')
config = json.loads(config_path.read_text(encoding='utf-8'))

random.seed(config['seed'])
torch.manual_seed(config['seed'])

print('실험 설정:', config)

실험 설정: {'seed': 42, 'n_per_class': 96, 'hidden_size': 16, 'batch_size': 32, 'learning_rate': 0.08, 'max_grad_norm': 0.5, 'warmup_start_factor': 0.2, 'warmup_steps': 4, 'max_steps': 8}


### 학습 설정 sanity check

설정 파일을 읽은 직후에는 값이 말이 되는 범위인지 먼저 확인합니다. 예를 들어 batch size가 예상보다 커지거나 learning rate가 한 자리 잘못 들어가면, 코드 자체에는 오류가 없어도 학습은 쉽게 불안정해집니다.

출력은 일부러 사람이 읽기 쉬운 문장으로 구성했습니다. 긴 실험을 시작하기 전에 seed, batch size, clipping 기준, warmup 길이를 눈으로 확인하는 과정은 사소해 보이지만 디버깅 시간을 크게 줄여 줍니다.

In [30]:
print('설정된 seed:', config['seed'])
print('클래스별 샘플 수 설정:', config['n_per_class'])
print('배치 크기:', config['batch_size'])
print('초기 learning rate:', config['learning_rate'])
print('gradient clipping 기준:', config['max_grad_norm'])

설정된 seed: 42
클래스별 샘플 수 설정: 96
배치 크기: 32
초기 learning rate: 0.08
gradient clipping 기준: 0.5


---
## 2. 실습용 데이터 생성

### 작은 데이터로 학습 이상 징후 관찰하기

두 개의 가우시안 분포에서 샘플을 만들어 간단한 이진 분류 문제를 구성합니다. 데이터가 작고 구조가 단순해야 loss와 gradient가 어떤 설정에서 어떻게 변하는지 빠르게 관찰할 수 있습니다.

복잡한 데이터셋에서 바로 안정화 기법을 실험하면 모델 문제인지, 데이터 문제인지, 설정 문제인지 구분하기 어렵습니다. 먼저 통제된 환경에서 관측 지표를 확인한 뒤 실제 모델에 적용하는 편이 안전합니다.

In [31]:
def make_toy_dataset(n_per_class=96):
    class0 = torch.randn(n_per_class, 2) * 0.7 + torch.tensor([-1.0, -1.0])
    class1 = torch.randn(n_per_class, 2) * 0.7 + torch.tensor([1.0, 1.0])
    x = torch.cat([class0, class1], dim=0)
    y = torch.cat([
        torch.zeros(n_per_class, dtype=torch.long),
        torch.ones(n_per_class, dtype=torch.long),
    ])
    order = torch.randperm(len(x))
    return x[order], y[order]

features, labels = make_toy_dataset(config['n_per_class'])
print('특성 텐서와 라벨 텐서 크기:', features.shape, labels.shape)
print('클래스별 샘플 수:', torch.bincount(labels).tolist())

특성 텐서와 라벨 텐서 크기: torch.Size([192, 2]) torch.Size([192])
클래스별 샘플 수: [96, 96]


---
## 3. 모델과 관측 함수 구성

### gradient norm을 함께 기록하는 이유

loss는 학습 결과를 보여 주지만, 업데이트가 왜 불안정해졌는지까지 항상 설명해 주지는 않습니다. gradient norm을 같이 보면 파라미터 업데이트의 크기가 갑자기 커졌는지, 현재 learning rate가 과한지, clipping이 실제로 필요한 상황인지 판단할 수 있습니다.

`grad_norm()`은 모든 파라미터의 gradient를 모아 하나의 스칼라로 요약합니다. 작은 MLP에서는 계산이 단순하지만, 실제 Transformer에서도 같은 아이디어로 전체 gradient scale을 관찰합니다. 중요한 점은 이 값을 optimizer step 전에 측정해야 한다는 것입니다. 그래야 실제 업데이트에 사용될 gradient 상태를 로그로 남길 수 있습니다.

In [32]:
class TinyClassifier(nn.Module):
    def __init__(self, hidden_size=16):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, hidden_size),
            nn.GELU(),
            nn.Linear(hidden_size, 2),
        )

    def forward(self, x):
        return self.net(x)


def grad_norm(parameters):
    total = torch.tensor(0.0)
    for param in parameters:
        if param.grad is not None:
            total += param.grad.detach().norm(2).pow(2)
    return total.sqrt().item()


def batch_iter(x, y, batch_size):
    order = torch.randperm(len(x))
    for start in range(0, len(x), batch_size):
        idx = order[start:start + batch_size]
        yield x[idx], y[idx]

---
## 4. 기본 학습 루프 실행

### 안정화 기법 없이 기준 로그 만들기

바로 clipping과 scheduler를 적용하면 무엇이 개선됐는지 확인하기 어렵습니다. 먼저 가장 단순한 학습 루프를 짧게 실행해 loss와 gradient norm의 기준선을 만듭니다.

이 루프는 일부러 기능을 최소화했습니다. `zero_grad()`, forward, loss 계산, backward, optimizer step만 포함해 학습의 기본 흐름을 보여 줍니다. 이후 안정화 루프와 비교하면 scheduler가 learning rate를 어떻게 바꾸는지, clipping 전후 gradient norm이 어떤 차이를 만드는지 더 분명하게 볼 수 있습니다.

In [33]:
def train_without_stabilizer(model, optimizer, x, y, max_steps=6):
    model.train()
    history = []
    step = 0
    for batch_x, batch_y in batch_iter(x, y, config['batch_size']):
        optimizer.zero_grad()
        logits = model(batch_x)
        loss = F.cross_entropy(logits, batch_y)
        loss.backward()
        norm = grad_norm(model.parameters())
        optimizer.step()
        history.append({'step': step, 'loss': float(loss.item()), 'grad_norm': norm, 'lr': optimizer.param_groups[0]['lr']})
        step += 1
        if step >= max_steps:
            break
    return history

baseline_model = TinyClassifier(config['hidden_size'])
baseline_optimizer = torch.optim.AdamW(baseline_model.parameters(), lr=config['learning_rate'])
baseline_history = train_without_stabilizer(baseline_model, baseline_optimizer, features, labels)
for row in baseline_history:
    print('기준 학습 로그:', row)

기준 학습 로그: {'step': 0, 'loss': 0.7597244381904602, 'grad_norm': 0.7379866242408752, 'lr': 0.08}
기준 학습 로그: {'step': 1, 'loss': 0.4681495726108551, 'grad_norm': 0.5608528256416321, 'lr': 0.08}
기준 학습 로그: {'step': 2, 'loss': 0.3003349304199219, 'grad_norm': 0.3901028633117676, 'lr': 0.08}
기준 학습 로그: {'step': 3, 'loss': 0.16055774688720703, 'grad_norm': 0.16128434240818024, 'lr': 0.08}
기준 학습 로그: {'step': 4, 'loss': 0.1428503394126892, 'grad_norm': 0.17712348699569702, 'lr': 0.08}
기준 학습 로그: {'step': 5, 'loss': 0.14722922444343567, 'grad_norm': 0.29488494992256165, 'lr': 0.08}


### 기준 학습 로그를 요약하기

step별 로그를 모두 읽는 대신, 비교에 필요한 값만 먼저 뽑아 봅니다. 시작 loss와 마지막 loss는 짧은 학습 동안 방향이 맞는지 보여 주고, 최대 gradient norm은 업데이트가 얼마나 거칠게 움직였는지 알려 줍니다.

실제 실험에서는 이런 요약값을 TensorBoard, WandB, MLflow 같은 도구에 함께 남기는 경우가 많습니다. 지금은 작은 딕셔너리 계산이지만, 안정화 설정을 바꿔 가며 비교할 때 기준점 역할을 합니다.

In [34]:
baseline_loss_start = baseline_history[0]['loss']
baseline_loss_end = baseline_history[-1]['loss']
baseline_max_grad_norm = max(row['grad_norm'] for row in baseline_history)

print('기준 학습 시작 loss:', round(baseline_loss_start, 4))
print('기준 학습 마지막 loss:', round(baseline_loss_end, 4))
print('기준 학습 최대 gradient norm:', round(baseline_max_grad_norm, 4))

기준 학습 시작 loss: 0.7597
기준 학습 마지막 loss: 0.1472
기준 학습 최대 gradient norm: 0.738


---
## 5. [TODO] 안정화 학습 step 구현

### [TODO] clipping과 scheduler를 학습 step에 연결하기

안정적인 학습 step은 순서가 중요합니다. gradient를 비운 뒤 forward와 loss 계산을 하고, backward로 gradient를 만든 다음, clipping을 적용하고 optimizer를 한 번 움직입니다. scheduler는 optimizer step 이후에 호출해야 현재 step의 업데이트가 끝난 뒤 다음 learning rate가 준비됩니다.

[TODO]에서는 이 흐름 중 gradient clipping과 optimizer/scheduler step을 직접 채웁니다. clipping 전 gradient norm과 clipping 후 gradient norm을 모두 기록하는 이유는, clipping이 단순히 호출됐는지가 아니라 실제로 업데이트 크기를 제한했는지 확인하기 위해서입니다.

완성해야 할 변수와 동작은 다음과 같습니다.

- `max_grad_norm`: `config['max_grad_norm']`에서 clipping 기준값을 꺼내 저장합니다.
- `clipped_norm`: `torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=max_grad_norm)`의 반환값입니다. 반환값은 clipping 적용 전 norm이므로 로그 해석에 사용할 수 있습니다.
- `optimizer_step_result`: `optimizer.step()`을 실행한 뒤, 실행 여부를 확인할 수 있도록 문자열 값을 넣습니다.
- `scheduler_step_result`: `scheduler.step()`을 실행한 뒤, 실행 여부를 확인할 수 있도록 문자열 값을 넣습니다.

In [ ]:
def train_with_stabilizer(model, optimizer, scheduler, x, y, max_steps=8):
    model.train()
    history = []
    step = 0
    for batch_x, batch_y in batch_iter(x, y, config['batch_size']):
        optimizer.zero_grad()
        logits = model(batch_x)
        loss = F.cross_entropy(logits, batch_y)
        loss.backward()

        norm_before = grad_norm(model.parameters())

        # [TODO 1] gradient clipping을 적용하세요.
        # 힌트: max_grad_norm을 기준으로 torch.nn.utils.clip_grad_norm_을 호출합니다.
        max_grad_norm = config['max_grad_norm']
        clipped_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=max_grad_norm)

        norm_after = grad_norm(model.parameters())

        # [TODO 2] optimizer와 scheduler를 올바른 순서로 진행하세요.
        optimizer.step()
        scheduler.step()
        optimizer_step_result = 'stepped'
        scheduler_step_result = 'stepped'

        history.append({
            'step': step,
            'loss': float(loss.item()),
            'grad_norm_before': norm_before,
            'grad_norm_after': norm_after,
            'lr': optimizer.param_groups[0]['lr'],
        })
        step += 1
        if step >= max_steps:
            break
    return history

---
## 6. checkpoint 저장과 복구

### 안정화 학습 실행하기

완성한 학습 루프를 실행하면서 loss, clipping 전후 gradient norm, learning rate를 함께 출력합니다. 이 조합은 긴 학습에서 가장 먼저 확인하는 기본 로그에 가깝습니다.

특히 learning rate는 scheduler 때문에 매 step 달라질 수 있으므로 loss만 따로 보면 해석이 어렵습니다. 특정 step에서 loss가 튀었다면 그때 gradient norm이 컸는지, clipping 이후에도 여전히 큰 값이 남았는지, learning rate가 어느 구간에 있었는지를 같이 봐야 합니다. 짧은 실습이지만 로그를 어떤 순서로 읽어야 하는지 연습하는 데 초점을 둡니다.

In [36]:
stable_model = TinyClassifier(config['hidden_size'])
stable_optimizer = torch.optim.AdamW(stable_model.parameters(), lr=config['learning_rate'])
stable_scheduler = torch.optim.lr_scheduler.LinearLR(
    stable_optimizer,
    start_factor=config['warmup_start_factor'],
    total_iters=config['warmup_steps'],
)

stable_history = train_with_stabilizer(
    stable_model,
    stable_optimizer,
    stable_scheduler,
    features,
    labels,
    max_steps=config['max_steps'],
)

for row in stable_history:
    print('안정화 학습 로그:', row)

안정화 학습 로그: {'step': 0, 'loss': 0.6880185604095459, 'grad_norm_before': 1.1711857318878174, 'grad_norm_after': 0.49999961256980896, 'lr': 0.032, 'optimizer_step_result': 'stepped', 'scheduler_step_result': 'stepped'}
안정화 학습 로그: {'step': 1, 'loss': 0.6003425121307373, 'grad_norm_before': 1.310875415802002, 'grad_norm_after': 0.49999967217445374, 'lr': 0.048, 'optimizer_step_result': 'stepped', 'scheduler_step_result': 'stepped'}
안정화 학습 로그: {'step': 2, 'loss': 0.5070140361785889, 'grad_norm_before': 0.6074323058128357, 'grad_norm_after': 0.49999916553497314, 'lr': 0.064, 'optimizer_step_result': 'stepped', 'scheduler_step_result': 'stepped'}
안정화 학습 로그: {'step': 3, 'loss': 0.35287928581237793, 'grad_norm_before': 0.4552507698535919, 'grad_norm_after': 0.4552507698535919, 'lr': 0.08, 'optimizer_step_result': 'stepped', 'scheduler_step_result': 'stepped'}
안정화 학습 로그: {'step': 4, 'loss': 0.20341648161411285, 'grad_norm_before': 0.3253597617149353, 'grad_norm_after': 0.3253597617149353, 'lr': 0

### 안정화 학습 로그와 기준 로그 비교하기

기준 학습과 안정화 학습의 요약값을 나란히 놓고 봅니다. 작은 데이터셋이라 loss 차이가 크게 보이지 않을 수도 있지만, gradient norm이 clipping 기준 근처에서 어떻게 제한되는지는 확인할 수 있습니다.

이 비교는 “코드가 실행됐다”에서 멈추지 않기 위한 단계입니다. 안정화 기법은 붙였다는 사실보다, 어떤 지표를 바꿨고 그 변화가 의도와 맞는지가 더 중요합니다.

In [37]:
stable_loss_start = stable_history[0]['loss']
stable_loss_end = stable_history[-1]['loss']
stable_max_grad_before = max(row['grad_norm_before'] for row in stable_history)
stable_max_grad_after = max(row['grad_norm_after'] for row in stable_history)

print('안정화 학습 시작 loss:', round(stable_loss_start, 4))
print('안정화 학습 마지막 loss:', round(stable_loss_end, 4))
print('clipping 전 최대 gradient norm:', round(stable_max_grad_before, 4))
print('clipping 후 최대 gradient norm:', round(stable_max_grad_after, 4))

안정화 학습 시작 loss: 0.688
안정화 학습 마지막 loss: 0.0832
clipping 전 최대 gradient norm: 1.3109
clipping 후 최대 gradient norm: 0.5


### checkpoint 파일로 학습 상태 저장하기

중단된 학습을 제대로 이어가려면 모델 가중치만 저장해서는 부족합니다. optimizer의 momentum 계열 상태, scheduler가 몇 step을 진행했는지, 현재 step 번호와 설정값이 함께 있어야 이전 실험의 흐름을 최대한 유지할 수 있습니다.

이 셀에서는 필요한 상태를 하나의 딕셔너리로 묶어 저장합니다. 저장 경로를 출력해 두는 이유는 복구 단계에서 어떤 파일을 읽는지 분명히 하기 위해서입니다. 실무에서는 checkpoint 이름에 epoch, step, validation score를 함께 넣어 여러 버전을 구분하는 경우가 많습니다.

In [38]:
checkpoint_dir = Path('checkpoints')
checkpoint_dir.mkdir(exist_ok=True)
checkpoint_path = checkpoint_dir / 'stable_training.pt'

torch.save({
    'model_state_dict': stable_model.state_dict(),
    'optimizer_state_dict': stable_optimizer.state_dict(),
    'scheduler_state_dict': stable_scheduler.state_dict(),
    'step': len(stable_history),
    'config': config,
}, checkpoint_path)

print('체크포인트 저장 경로:', checkpoint_path)
print('저장된 학습 step 수:', len(stable_history))

체크포인트 저장 경로: checkpoints/stable_training.pt
저장된 학습 step 수: 6


### checkpoint에 들어간 항목 확인하기

파일을 저장한 직후 곧바로 다시 열어 key 목록을 확인합니다. 저장 자체가 성공했더라도, 나중에 복구에 필요한 `optimizer_state_dict`나 `scheduler_state_dict`가 빠져 있으면 이어 학습 단계에서 문제가 드러납니다.

작은 확인이지만 checkpoint 설계에서는 중요한 습관입니다. 복구에 필요한 정보가 충분한지 저장 시점에 검증해야, 장시간 학습이 끝난 뒤에야 누락을 발견하는 일을 피할 수 있습니다.

In [27]:
checkpoint_preview = torch.load(checkpoint_path, weights_only=False)
print('체크포인트에 저장된 항목:', sorted(checkpoint_preview.keys()))
print('체크포인트에 저장된 설정 seed:', checkpoint_preview['config']['seed'])

체크포인트에 저장된 항목: ['config', 'model_state_dict', 'optimizer_state_dict', 'scheduler_state_dict', 'step']
체크포인트에 저장된 설정 seed: 42


### checkpoint에서 학습 상태 복구하기

복구할 때는 저장 당시와 같은 구조의 모델, optimizer, scheduler를 먼저 만들어야 합니다. 그다음 checkpoint 안의 state dict를 각각 로드해야 모델 가중치뿐 아니라 optimizer 내부 상태와 learning rate 진행 상태까지 이어집니다.

모델만 복구하면 겉으로는 정상처럼 보일 수 있지만, optimizer가 새로 초기화되어 이전 업데이트 흐름과 달라질 수 있습니다. 특히 AdamW 계열 optimizer는 이동 평균 상태를 갖고 있기 때문에, 긴 학습에서는 이 차이가 이어 학습 결과에 영향을 줍니다. 마지막 출력은 저장된 step과 복구된 learning rate를 확인해 다음 학습을 이어갈 준비가 되었는지 점검합니다.

In [39]:
restored_model = TinyClassifier(config['hidden_size'])
restored_optimizer = torch.optim.AdamW(restored_model.parameters(), lr=config['learning_rate'])
restored_scheduler = torch.optim.lr_scheduler.LinearLR(
    restored_optimizer,
    start_factor=config['warmup_start_factor'],
    total_iters=config['warmup_steps'],
)

state = torch.load(checkpoint_path, weights_only=False)
restored_model.load_state_dict(state['model_state_dict'])
restored_optimizer.load_state_dict(state['optimizer_state_dict'])
restored_scheduler.load_state_dict(state['scheduler_state_dict'])

print('복구된 학습 step:', state['step'])
print('복구된 learning rate:', restored_optimizer.param_groups[0]['lr'])

복구된 학습 step: 6
복구된 learning rate: 0.08
